# 04 — XGBoost · Overload Prediction

**Input:** `hive_metastore.gold.gold_features` (pre-scaled, with `split` column)  
**Tracking:** MLflow experiment `Transformer_Overload`

### Notebook structure

| Section | Description |
|---|---|
| 1 | Configuration & imports |
| 2 | Load data (train/test from `split` column) |
| 3 | Validation split for early stopping |
| 4 | Feature assembly pipeline |
| 5 | Hyperparameter grid search (with MLflow) |
| 6 | Best model evaluation on test set |
| 7 | Threshold sweep |
| 8 | Confusion matrix & curves |
| 9 | Feature importance (individual + grouped) |
| 10 | Summary |

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/03_gold_features/00_Evaluation

## 1 · Configuration & imports

In [0]:
import builtins
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, FeatureHasher
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.functions import vector_to_array

import warnings
warnings.filterwarnings("ignore")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
SRC_TABLE    = "hive_metastore.gold.gold_features"
ID_COL       = "ID_prefix"
TS_COL       = "DATE"
LABEL_COL    = "label_4h"            # change to "label_24h" for 24 h horizon
MODEL_NAME   = "XGBoost"
EXPERIMENT   = "/Users/daniel.branco@cgi.com/Transformer_Overload"
SEED         = 42

# ── Feature groups (must match 02_gold_features) ─────────────────────────────
SIGNAL_COLS = ["current", "voltage"]

LOAD_RATIO_COLS = ["load_ratio_c", "load_ratio_v"]

ROLLING_COLS = [
    f"{s}_{stat}_{w}"
    for s in SIGNAL_COLS
    for stat in ["mean", "std", "max"]
    for w in ["1h", "1d", "7d"]
]

LAG_COLS = [
    f"{s}_lag_{l}"
    for s in SIGNAL_COLS
    for l in ["15m", "1h", "1d"]
]

WEATHER_RAW_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "precipitacao_horaria_mm",
    "humidade_relativa_media_horaria_percent",
    "velocidade_do_vento_media_horaria_m_per_s",
]

WEATHER_DERIVED_COLS = ["temp_mean_1d", "temp_mean_7d", "precip_sum_1d"]

TEMPORAL_COLS = ["hour", "day_of_week", "month", "is_weekend"]

EVENT_COLS = ["events_15m_cnt"]

NUMERIC_FEATURE_COLS = (
    LOAD_RATIO_COLS
    + ROLLING_COLS
    + LAG_COLS
    + WEATHER_RAW_COLS
    + WEATHER_DERIVED_COLS
    + TEMPORAL_COLS
    + EVENT_COLS
)

CAT_COLS = [ID_COL, "CONCELHO"]

# Hash buckets for ID_prefix (1000+ unique values)
N_HASH_BUCKETS = 256

# ── Sets used by classify_feature() in 00_Evaluation ─────────────────────────
WEATHER_RAW_SET     = set(WEATHER_RAW_COLS)
WEATHER_DERIVED_SET = set(WEATHER_DERIVED_COLS)
LOAD_RATIO_SET      = set(LOAD_RATIO_COLS)
TEMPORAL_SET        = set(TEMPORAL_COLS)
EVENT_SET           = set(EVENT_COLS)

# ── Validation split for early stopping ──────────────────────────────────────
# Last 20% of training rows (chronologically) used as validation
VAL_FRAC = 0.2

print(f"Numeric features : {len(NUMERIC_FEATURE_COLS)}")
print(f"Categorical cols : {CAT_COLS}")
print(f"Hash buckets     : {N_HASH_BUCKETS}")
print(f"Label            : {LABEL_COL}")

## 2 · Load data

In [0]:
df = spark.read.table(SRC_TABLE)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")

keep_cols = NUMERIC_FEATURE_COLS + [LABEL_COL] + CAT_COLS + [TS_COL, "split"]

existing = set(df.columns)
missing = set(NUMERIC_FEATURE_COLS) - existing
if missing:
    print(f"⚠️  Missing columns (removed): {missing}")
    NUMERIC_FEATURE_COLS[:] = [c for c in NUMERIC_FEATURE_COLS if c in existing]

keep_cols = [c for c in keep_cols if c in existing]
df_clean = df.select(keep_cols).dropna()
print(f"After dropna: {df_clean.count():,} rows")

In [0]:
train_df = df_clean.filter(F.col("split") == "train")
test_df  = df_clean.filter(F.col("split") == "test")

print(f"Train : {train_df.count():,} rows")
print(f"Test  : {test_df.count():,} rows")

# Class imbalance ratio
n_train = train_df.count()
n_pos   = train_df.filter(F.col(LABEL_COL) == 1).count()
n_neg   = n_train - n_pos
scale_pos_weight = float(n_neg) / builtins.max(n_pos, 1)

print(f"\nTrain: {n_pos:,} pos ({100*n_pos/n_train:.2f}%)  |  {n_neg:,} neg")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

## 3 · Validation split for early stopping

XGBoost uses a `validation_indicator_col` (boolean column where `True` = validation row).  
We take the last 20% of training rows chronologically as validation.

In [0]:
# Find the chronological cutoff for validation (last 20% of training rows)
import datetime

total_train = train_df.count()

# Get min/max dates
date_range = train_df.agg(F.min(TS_COL).alias("min_dt"), F.max(TS_COL).alias("max_dt")).first()
min_dt = date_range["min_dt"]
max_dt = date_range["max_dt"]

# Cutoff = 80% of the way through the date range
total_seconds = (max_dt - min_dt).total_seconds()
val_cutoff = min_dt + datetime.timedelta(seconds=total_seconds * (1 - VAL_FRAC))
print(f"Validation cutoff: {val_cutoff}")

# Add is_val boolean column
train_w = train_df.withColumn(
    "is_val",
    F.col(TS_COL) >= F.lit(val_cutoff)
)

n_train_actual = train_w.filter(~F.col("is_val")).count()
n_val = train_w.filter(F.col("is_val")).count()
print(f"Train (actual) : {n_train_actual:,}")
print(f"Validation     : {n_val:,}")

train_w.cache()
test_df.cache()
print(f"\nCached ✅")

## 4 · Feature assembly pipeline

XGBoost is tree-based — scaling doesn't matter, but the data is pre-scaled anyway.  
Categoricals: FeatureHasher for `ID_prefix` (1000+ values), OHE for `CONCELHO` (18 values).

In [0]:
def make_xgb_pipeline(
    learning_rate=0.06,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.6,
    num_round=1200,
    reg_lambda=2.0,
    min_child_weight=5,
    gamma=1.0,
):
    # Hash both categoricals together
    hasher = FeatureHasher(
        inputCols=CAT_COLS,
        outputCol="cat_hashed",
        numFeatures=N_HASH_BUCKETS,
        categoricalCols=CAT_COLS,
    )

    feat_assembler = VectorAssembler(
        inputCols=["cat_hashed"] + NUMERIC_FEATURE_COLS,
        outputCol="features",
        handleInvalid="skip",
    )

    xgb = SparkXGBClassifier(
        features_col="features",
        label_col=LABEL_COL,
        prediction_col="prediction",
        probability_col="probability",
        raw_prediction_col="rawPrediction",
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        num_round=num_round,
        reg_lambda=reg_lambda,
        min_child_weight=min_child_weight,
        gamma=gamma,
        scale_pos_weight=float(scale_pos_weight),
        eval_metric="aucpr",
        tree_method="hist",
        validation_indicator_col="is_val",
        early_stopping_rounds=100,
        num_workers=builtins.max(1, spark.sparkContext.defaultParallelism // 2),
    )

    return Pipeline(stages=[hasher, feat_assembler, xgb])

## 5 · Hyperparameter grid search (with MLflow)

Each configuration gets its own nested MLflow run.  
Primary selection metric: **AUPRC** (evaluated on the held-out test set).

In [0]:
# (learning_rate, max_depth, colsample_bytree, min_child_weight, gamma, reg_lambda, num_round)
PARAM_GRID = [
    (0.10, 6, 0.6, 5,  1.0, 2.0, 800),
    (0.06, 6, 0.6, 10, 1.0, 2.0, 1200),
    (0.05, 6, 0.8, 8,  0.0, 3.0, 1500),
    (0.04, 8, 0.6, 12, 1.0, 3.0, 2000),
    (0.06, 8, 0.8, 5,  0.5, 1.0, 1000),
    (0.03, 6, 0.7, 10, 1.0, 2.0, 2500),
]

In [0]:
mlflow.set_experiment(EXPERIMENT)

best = {"auprc": -1, "run_id": None, "model": None, "params": None}

with mlflow.start_run(run_name=f"{MODEL_NAME}_{LABEL_COL}") as parent_run:

    mlflow.log_param("model_type", MODEL_NAME)
    mlflow.log_param("label", LABEL_COL)
    mlflow.log_param("n_numeric_features", len(NUMERIC_FEATURE_COLS))
    mlflow.log_param("cat_cols", str(CAT_COLS))
    mlflow.log_param("scale_pos_weight", float(f"{scale_pos_weight:.4f}"))
    mlflow.log_param("n_hash_buckets", N_HASH_BUCKETS)

    for lr, md, csbt, mcw, gma, rl2, nr in PARAM_GRID:
        run_name = f"lr={lr}_md={md}_csbt={csbt}_mcw={mcw}"

        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            mlflow.log_param("learning_rate", lr)
            mlflow.log_param("max_depth", md)
            mlflow.log_param("colsample_bytree", csbt)
            mlflow.log_param("min_child_weight", mcw)
            mlflow.log_param("gamma", gma)
            mlflow.log_param("reg_lambda", rl2)
            mlflow.log_param("num_round", nr)

            print(f"\n▶ Training: lr={lr} md={md} csbt={csbt} mcw={mcw} gamma={gma} regL2={rl2} rounds={nr}")

            pipe = make_xgb_pipeline(
                learning_rate=lr, max_depth=md,
                colsample_bytree=csbt, min_child_weight=mcw,
                gamma=gma, reg_lambda=rl2, num_round=nr,
            )
            model = pipe.fit(train_w)

            # Predict on test
            preds = model.transform(test_df)
            preds = preds.withColumn("prob_pos", vector_to_array("probability")[1])

            local = preds.select(F.col(LABEL_COL).cast("int"), "prob_pos").toPandas()
            y_true = local[LABEL_COL].values
            y_prob = local["prob_pos"].values

            metrics = evaluate_binary(y_true, y_prob, threshold=0.5, title=f"{MODEL_NAME} | {run_name}")
            log_evaluation_to_mlflow(metrics, y_true, y_prob, threshold=0.5, prefix="test")

            if metrics["AUPRC"] > best["auprc"]:
                best = {
                    "auprc":  metrics["AUPRC"],
                    "run_id": child_run.info.run_id,
                    "model":  model,
                    "params": (lr, md, csbt, mcw, gma, rl2, nr),
                    "metrics": metrics,
                }

    mlflow.log_param("best_config", str(best["params"]))
    mlflow.log_metric("best_AUPRC", best["auprc"])
    mlflow.log_metric("best_AUROC", best["metrics"]["AUROC"])
    mlflow.log_metric("best_f1", best["metrics"]["f1"])
    mlflow.spark.log_model(best["model"], artifact_path="best_xgb_model")

print(f"\n🏆 Best config: {best['params']}")
print(f"   AUPRC={best['auprc']:.4f}  |  Run ID: {best['run_id']}")

## 6 · Best model — full evaluation on test set

In [0]:
best_model = best["model"]
preds_best = best_model.transform(test_df)
preds_best = preds_best.withColumn("prob_pos", vector_to_array("probability")[1])

local_best = preds_best.select(F.col(LABEL_COL).cast("int"), "prob_pos", "prediction").toPandas()
y_true = local_best[LABEL_COL].values
y_prob = local_best["prob_pos"].values

final_metrics = evaluate_binary(y_true, y_prob, threshold=0.5, title=f"FINAL TEST — {MODEL_NAME} ({LABEL_COL})")

## 7 · Threshold sweep

In [0]:
sweep_results = threshold_sweep(y_true, y_prob)
display(spark.createDataFrame(sweep_results).orderBy(F.desc("f1")))

In [0]:
best_thr_row = builtins.max(sweep_results, key=lambda r: r["f1"])
optimal_threshold = best_thr_row["threshold"]
print(f"Optimal threshold (max F1): {optimal_threshold}")

optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

## 8 · Confusion matrix & curves

In [0]:
fig = plot_confusion_matrix(y_true, (y_prob >= 0.5).astype(int), title=f"{MODEL_NAME} — CM @ 0.5")
display(fig); plt.close(fig)

fig = plot_confusion_matrix(y_true, (y_prob >= optimal_threshold).astype(int), title=f"{MODEL_NAME} — CM @ {optimal_threshold}")
display(fig); plt.close(fig)

In [0]:
fig = plot_roc_curve(y_true, y_prob, title=f"{MODEL_NAME} — ROC")
display(fig); plt.close(fig)

In [0]:
fig = plot_pr_curve(y_true, y_prob, title=f"{MODEL_NAME} — Precision-Recall")
display(fig); plt.close(fig)

## 9 · Feature importance (individual + grouped)

For XGBoost, importance = **gain** (total reduction in loss contributed by each feature).  
Grouping and plotting functions come from `00_Evaluation`.

In [0]:
# ── XGBoost-specific: extract gain-based importance ──────────────────────────
from xgboost import XGBClassifier

# Get the native XGBoost booster from the Spark model
xgb_spark_model = None
for stage in best_model.stages:
    if hasattr(stage, "get_booster"):
        xgb_spark_model = stage
        break

assert xgb_spark_model is not None, "Could not find XGBoost stage in pipeline"

booster = xgb_spark_model.get_booster()
importance_dict = booster.get_score(importance_type="gain")

print(f"Features with non-zero gain: {len(importance_dict)}")

# ── Recover feature names from VectorAssembler metadata ──────────────────────
pred_one = best_model.transform(test_df.limit(1))
feat_meta = pred_one.schema["features"].metadata

feat_names = []
if "ml_attr" in feat_meta and "attrs" in feat_meta["ml_attr"]:
    for attr_type in feat_meta["ml_attr"]["attrs"]:
        for attr in feat_meta["ml_attr"]["attrs"][attr_type]:
            feat_names.append((attr["idx"], attr["name"]))
    feat_names.sort(key=lambda x: x[0])
    feat_names = [n for _, n in feat_names]
else:
    # Fallback: use booster feature names (f0, f1, ...)
    n_features = builtins.max(int(k.replace("f", "")) for k in importance_dict.keys()) + 1
    feat_names = [f"f{i}" for i in range(n_features)]

print(f"Feature names: {len(feat_names)}")

# ── Map booster importance (f0, f1, ...) to named features ───────────────────
importances = np.zeros(len(feat_names))
for key, gain in importance_dict.items():
    idx = int(key.replace("f", ""))
    if idx < len(importances):
        importances[idx] = float(gain)

print(f"Total gain: {float(np.sum(importances)):.2f}")

In [0]:
fig_top = plot_top_features(feat_names, importances, top_n=20, title=f"{MODEL_NAME} — Top 20 by Gain", xlabel="Gain")
display(fig_top)

# Table view
imp_rows = [
    (feat_names[i], float(importances[i]), classify_feature(feat_names[i]))
    for i in range(len(feat_names)) if importances[i] > 0
]
imp_df = spark.createDataFrame(imp_rows, ["feature", "gain", "group"])
display(imp_df.orderBy(F.desc("gain")).limit(20))

In [0]:
grp = grouped_importance(feat_names, importances)
fig_grp = plot_grouped_importance(grp, title=f"{MODEL_NAME} — Feature Group Importance", xlabel="Sum Gain")
display(fig_grp)

grp_rows = [(g, d["n_dims"], float(d["sum"]), float(d["mean"])) for g, d in grp.items()]
display(spark.createDataFrame(grp_rows, ["group", "n_dims", "sum_gain", "mean_gain"]).orderBy(F.desc("sum_gain")))

In [0]:
import pandas as pd

with mlflow.start_run(run_id=best["run_id"]):
    mlflow.log_figure(fig_top, "feature_importance_top20.png")
    mlflow.log_figure(fig_grp, "feature_importance_grouped.png")
    mlflow.log_table(
        pd.DataFrame([dict(group=g, **d) for g, d in grp.items()]),
        artifact_file="grouped_feature_importance.json",
    )

plt.close(fig_top)
plt.close(fig_grp)
print("✅ Feature importance logged to MLflow")

## 10 · Summary

In [0]:
lr, md, csbt, mcw, gma, rl2, nr = best["params"]

print("\n" + "="*60)
print(f"  MODEL SUMMARY — {MODEL_NAME}")
print("="*60)
print(f"  Label            : {LABEL_COL}")
print(f"  scale_pos_weight : {scale_pos_weight:.4f}")
print(f"  Best config:")
print(f"    learning_rate    : {lr}")
print(f"    max_depth        : {md}")
print(f"    colsample_bytree : {csbt}")
print(f"    min_child_weight : {mcw}")
print(f"    gamma            : {gma}")
print(f"    reg_lambda       : {rl2}")
print(f"    num_round        : {nr}")
print(f"  ─────────────────────────────────")
print(f"  Test AUROC     : {final_metrics['AUROC']:.4f}")
print(f"  Test AUPRC     : {final_metrics['AUPRC']:.4f}")
print(f"  Test F1 @0.5   : {final_metrics['f1']:.4f}")
print(f"  Test MCC @0.5  : {final_metrics['mcc']:.4f}")
print(f"  Optimal thr    : {optimal_threshold}")
print(f"  F1 @opt thr    : {optimal_metrics['f1']:.4f}")
print(f"  ─────────────────────────────────")
print(f"  MLflow run     : {best['run_id']}")
print("="*60)

train_w.unpersist()
test_df.unpersist()
print("\n✅ Done. Caches released.")